# 🚦 Hệ thống Hỏi Đáp Luật Giao Thông Việt Nam
**Model:** Qwen2.5-1.5B-Instruct + QLoRA + Hybrid RAG (Dataset + FAISS)

## Chiến lược tối ưu:
- **Tầng 1:** Tìm trong dataset QA (exact + semantic match) → nhanh nhất, chính xác nhất
- **Tầng 2:** Tra cứu bảng phạt theo loại xe từ knowledge base → đúng mức phạt + nghị định
- **Tầng 3:** FAISS + model → fallback cho câu hỏi phức tạp
- Model fine-tune giữ nguyên cho yêu cầu đề tài (so sánh 4 cấu hình A/B/C/D)


## Cell 1 — Cài đặt thư viện

In [1]:
!pip install -q -U transformers==4.46.3 huggingface_hub==0.26.5
!pip install -q datasets peft accelerate bitsandbytes trl
!pip install -q langchain langchain-community sentence-transformers faiss-cpu
!pip install -q rouge_score evaluate bert-score absl-py nltk
!pip install -q fastapi uvicorn pydantic pyngrok
print('Cài đặt xong!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 124.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.5 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3 — Chuẩn bị Dataset

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    df = pd.read_csv('dataset.csv', encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv('dataset.csv', encoding='latin1')
    df['Question'] = df['Question'].apply(lambda x: str(x).encode('latin1').decode('utf-8'))
    df['Answer']   = df['Answer'].apply(lambda x: str(x).encode('latin1').decode('utf-8'))

df = df.rename(columns={'Question': 'input', 'Answer': 'output'})
df['instruction'] = 'Trả lời câu hỏi luật giao thông Việt Nam'
df = df[['instruction', 'input', 'output']].dropna()

print(f'Tổng số cặp QA: {len(df)}')
print(df.head(2))

df_train, df_test = train_test_split(df, test_size=0.1, random_state=42)
df_train.to_csv('dataset_train.csv', index=False, encoding='utf-8')
df_test.to_csv('dataset_test.csv',   index=False, encoding='utf-8')
print(f'Train: {len(df_train)} | Test: {len(df_test)}')

Tổng số cặp QA: 350
                                instruction  \
0  Trả lời câu hỏi luật giao thông Việt Nam   
1  Trả lời câu hỏi luật giao thông Việt Nam   

                                               input  \
0  Khi tham gia giao thông trên Quốc lộ 1A, các ô...   
1  Tôi đang sử dụng 01 chiếc xe máy không chính c...   

                                              output  
0  1. Nội dung câu hỏi mà công dân nêu phải đặt t...  
1  Về thủ tục sang tên xe, đề nghị bạn căn cứ the...  
Train: 315 | Test: 35


## Cell 4 — Format Dataset (ChatML)

In [4]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files={'train': 'dataset_train.csv'})

def format_chatml(example):
    text = (
        '<|im_start|>system\n'
        'Bạn là chuyên gia tư vấn luật giao thông Việt Nam. '
        'Trả lời chính xác bằng tiếng Việt, nêu rõ mức phạt theo từng loại phương tiện '
        'và căn cứ nghị định cụ thể.'
        '<|im_end|>\n'
        f'<|im_start|>user\n{example["input"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{example["output"]}<|im_end|>'
    )
    return {'text': text}

dataset = dataset.map(format_chatml)
print('Ví dụ format:')
print(dataset['train'][0]['text'][:300])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/315 [00:00<?, ? examples/s]

Ví dụ format:
<|im_start|>system
Bạn là chuyên gia tư vấn luật giao thông Việt Nam. Trả lời chính xác bằng tiếng Việt, nêu rõ mức phạt theo từng loại phương tiện và căn cứ nghị định cụ thể.<|im_end|>
<|im_start|>user
Lái xe ô tô không thắt dây đai an toàn khi xe chạy trên đường bị phạt bao nhiêu?<|im_end|>
<|im_s


## Cell 5 — Load Qwen2.5-1.5B-Instruct

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.clean_up_tokenization_spaces = False

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True
)
model.config.use_cache = False
print(f'Load xong {MODEL_NAME}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Load xong Qwen/Qwen2.5-1.5B-Instruct


## Cell 6 — LoRA Config

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


## Cell 7 — Fine-tune

In [7]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir='./qwen25_finetuned',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    fp16=False, bf16=False,
    logging_steps=10, save_steps=200,
    max_length=256,
    dataset_text_field='text',
    packing=False, report_to='none'
)
trainer = SFTTrainer(model=model, train_dataset=dataset['train'], args=sft_config)
print('Bắt đầu fine-tune...')
trainer.train()
print('Fine-tune xong!')

Adding EOS to train dataset:   0%|          | 0/315 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/315 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Bắt đầu fine-tune...


Step,Training Loss
10,2.465861
20,2.153029
30,1.696947
40,1.323737
50,1.027925
60,0.915998
70,0.875544
80,0.839771


Fine-tune xong!


## Cell 8 — Lưu Model

In [8]:
model.save_pretrained('finetuned_qwen25')
tokenizer.save_pretrained('finetuned_qwen25')
print('Đã lưu model vào ./finetuned_qwen25')
# Upload HuggingFace Hub
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub('your-username/qwen25-luat-giaothong')

Đã lưu model vào ./finetuned_qwen25


## Cell 9 — Build Hybrid RAG
### Tầng 1: FAISS trên dataset QA | Tầng 2: FAISS trên knowledge base | Tầng 3: Bảng tra cứu


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
import re
import sys

embedding = HuggingFaceEmbeddings(
    model_name='keepitreal/vietnamese-sbert',
    model_kwargs={'device': 'cuda'}
)

# ── Tầng 1: FAISS trên dataset QA ────────────────────────
print("Đang xử lý dataset QA...")
try:
    df_all = pd.read_csv('dataset.csv', encoding='utf-8')
except UnicodeDecodeError:
    df_all = pd.read_csv('dataset.csv', encoding='latin1')

# Cố gắng tìm và đổi tên các cột có chứa từ khóa (không phân biệt hoa thường)
col_mapping = {}
for col in df_all.columns:
    if 'question' in col.lower() or 'input' in col.lower():
        col_mapping[col] = 'input'
    elif 'answer' in col.lower() or 'output' in col.lower():
        col_mapping[col] = 'output'

df_all = df_all.rename(columns=col_mapping).dropna(subset=['input', 'output'])

if len(df_all) == 0:
    print("LỖI NGHIÊM TRỌNG: File dataset.csv của bạn không có dữ liệu, hoặc không có các cột Question/Answer. Hãy kiểm tra lại file!")
    sys.exit() # Dừng chương trình để bạn kiểm tra

qa_docs = [
    Document(
        page_content=str(row['input']),
        metadata={'answer': str(row['output']), 'source': 'dataset'}
    )
    for _, row in df_all.iterrows()
]

# Bây giờ qa_docs chắc chắn không rỗng
db_qa = FAISS.from_documents(qa_docs, embedding)
db_qa.save_local('faiss_qa')
print(f'FAISS QA: {len(qa_docs)} câu hỏi')

# ── Tầng 2: FAISS trên knowledge base ────────────────────
print("Đang xử lý Knowledge Base...")
try:
    with open('knowledge_base_clean.txt', 'r', encoding='utf-8') as f:
        kb_text = f.read()
except FileNotFoundError:
    print("LỖI NGHIÊM TRỌNG: Không tìm thấy file 'knowledge_base_clean.txt'. Bạn đã tải file này lên chưa?")
    sys.exit()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)
kb_chunks = splitter.split_text(kb_text)

if not kb_chunks:
    print("LỖI NGHIÊM TRỌNG: File 'knowledge_base_clean.txt' bị trống!")
    sys.exit()

db_kb = FAISS.from_texts(kb_chunks, embedding)
db_kb.save_local('faiss_kb')
print(f'FAISS KB: {len(kb_chunks)} chunks')

# ── Tầng 3: Bảng tra cứu mức phạt theo loại xe ──────────
print("Đang xây dựng Bảng tra cứu (Tầng 3)...")
NGHI_DINH = '168/2024/NĐ-CP'

d6_start = kb_text.find('Điều 6. Xử phạt')
d7_start = kb_text.find('Điều 7. Xử phạt')
d8_start = kb_text.find('Điều 8. Xử phạt')
d18_start = kb_text.find('Điều 18. Xử phạt')
d19_start = kb_text.find('Điều 19.', d18_start) if d18_start > 0 else len(kb_text)

TEXT_OTO  = kb_text[d6_start:d7_start] if d6_start != -1 else ""
TEXT_XM   = kb_text[d7_start:d8_start] if d7_start != -1 else ""
TEXT_GPLX = kb_text[d18_start:d19_start] if d18_start > 0 else ''
def extract_penalty_map(dieu_text, loai_xe_label):
    if not dieu_text: return []
    result = []
    khoans = list(re.finditer(
        r'\n(\d+)\. Phạt tiền từ ([\d\.\,]+) đồng đến ([\d\.\,]+) đồng', dieu_text
    ))
    for i, m in enumerate(khoans):
        start = m.start()
        end = khoans[i+1].start() if i+1 < len(khoans) else start + 3000
        block = dieu_text[start:end]
        points = re.findall(r'([a-zđ]\)) ([^\n;]{10,120})', block)
        for _, point_text in points:
            result.append({
                'loai_xe': loai_xe_label,
                'hanh_vi': point_text.strip(),
                'min': m.group(2),
                'max': m.group(3),
                'khoan': m.group(1),
            })
    return result

PENALTY_TABLE = (
    extract_penalty_map(TEXT_OTO, 'Ô tô') +
    extract_penalty_map(TEXT_XM,  'Xe máy/mô tô') +
    extract_penalty_map(TEXT_GPLX, 'Giấy phép lái xe')
)

if len(PENALTY_TABLE) == 0:
    print("CẢNH BÁO: Bảng tra cứu trống. Có thể định dạng văn bản luật đã thay đổi hoặc file knowledge_base_clean.txt không chứa Điều 6, Điều 7.")
else:
    print(f'Bảng tra cứu: {len(PENALTY_TABLE)} mức phạt theo loại xe')

/tmp/ipykernel_2640/600591230.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Đang xử lý dataset QA...
FAISS QA: 350 câu hỏi
Đang xử lý Knowledge Base...
FAISS KB: 415 chunks
Đang xây dựng Bảng tra cứu (Tầng 3)...
Bảng tra cứu: 181 mức phạt theo loại xe


## Cell 10 — Load lại FAISS (nếu đã build rồi)

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import pandas as pd
import re
import sys

embedding = HuggingFaceEmbeddings(
    model_name='keepitreal/vietnamese-sbert',
    model_kwargs={'device': 'cuda'}
)

print("Đang tải FAISS QA...")
try:
    db_qa = FAISS.load_local('faiss_qa', embedding, allow_dangerous_deserialization=True)
    print(f'Đã tải FAISS QA: {db_qa.index.ntotal} câu hỏi')
except Exception as e:
    print(f"Không thể tải FAISS QA. Lỗi: {e}")
    print("Tiến hành tạo mới FAISS QA...")

    try:
        df_all = pd.read_csv('dataset.csv', encoding='utf-8')
    except UnicodeDecodeError:
        df_all = pd.read_csv('dataset.csv', encoding='latin1')
    except FileNotFoundError:
        print("LỖI: Không tìm thấy file 'dataset.csv'. Vui lòng kiểm tra lại.")
        sys.exit()

    # Rename columns flexibly
    col_mapping = {}
    for col in df_all.columns:
        col_lower = col.lower()
        if 'question' in col_lower or 'input' in col_lower or 'câu hỏi' in col_lower:
            col_mapping[col] = 'input'
        elif 'answer' in col_lower or 'output' in col_lower or 'trả lời' in col_lower:
            col_mapping[col] = 'output'

    if not col_mapping:
         print(f"LỖI: Không tìm thấy cột 'Question/Input' hoặc 'Answer/Output' trong 'dataset.csv'. Các cột hiện có: {df_all.columns.tolist()}")
         sys.exit()

    df_all = df_all.rename(columns=col_mapping).dropna(subset=['input', 'output'])

    if len(df_all) == 0:
        print("LỖI: File 'dataset.csv' không có dữ liệu hợp lệ sau khi làm sạch. Vui lòng kiểm tra lại file.")
        sys.exit()

    qa_docs = [
        Document(
            page_content=str(row['input']),
            metadata={'answer': str(row['output']), 'source': 'dataset'}
        )
        for _, row in df_all.iterrows()
    ]

    db_qa = FAISS.from_documents(qa_docs, embedding)
    db_qa.save_local('faiss_qa')
    print(f'Tạo mới FAISS QA: {len(qa_docs)} câu hỏi')

print("\nĐang tải FAISS KB...")
try:
    db_kb = FAISS.load_local('faiss_kb', embedding, allow_dangerous_deserialization=True)
    print(f'Đã tải FAISS KB: {db_kb.index.ntotal} chunks')
except Exception as e:
    print(f" Không thể tải FAISS KB. Lỗi: {e}")
    print("Vui lòng chạy lại Cell 9 để tạo FAISS KB.")
    sys.exit()

print("\nĐang xây dựng Bảng tra cứu (Tầng 3)...")
try:
    with open('knowledge_base_clean.txt', 'r', encoding='utf-8') as f:
        kb_text = f.read()
except FileNotFoundError:
     print("LỖI: Không tìm thấy file 'knowledge_base_clean.txt'. Vui lòng kiểm tra lại.")
     sys.exit()

NGHI_DINH = '168/2024/NĐ-CP'

d6_start = kb_text.find('Điều 6. Xử phạt')
d7_start = kb_text.find('Điều 7. Xử phạt')
d8_start = kb_text.find('Điều 8. Xử phạt')
d18_start = kb_text.find('Điều 18. Xử phạt')
d19_start = kb_text.find('Điều 19.', d18_start) if d18_start > 0 else len(kb_text)

TEXT_OTO  = kb_text[d6_start:d7_start] if d6_start != -1 else ""
TEXT_XM   = kb_text[d7_start:d8_start] if d7_start != -1 else ""

def extract_penalty_map(dieu_text, loai_xe_label):
    if not dieu_text: return []
    result = []
    khoans = list(re.finditer(
        r'\n(\d+)\. Phạt tiền từ ([\d\.\,]+) đồng đến ([\d\.\,]+) đồng', dieu_text
    ))
    for i, m in enumerate(khoans):
        start = m.start()
        end = khoans[i+1].start() if i+1 < len(khoans) else start + 3000
        block = dieu_text[start:end]
        points = re.findall(r'([a-zđ]\)) ([^\n;]{10,120})', block)
        for _, point_text in points:
            result.append({
                'loai_xe': loai_xe_label,
                'hanh_vi': point_text.strip(),
                'min': m.group(2),
                'max': m.group(3),
                'khoan': m.group(1),
            })
    return result

PENALTY_TABLE = (
    extract_penalty_map(TEXT_OTO, 'Ô tô') +
    extract_penalty_map(TEXT_XM,  'Xe máy/mô tô')
)

if not PENALTY_TABLE:
    print("⚠️ CẢNH BÁO: Bảng tra cứu trống. Có thể định dạng văn bản luật đã thay đổi hoặc file knowledge_base_clean.txt không chứa Điều 6, Điều 7.")
else:
    print(f'Bảng tra cứu: {len(PENALTY_TABLE)} mức phạt theo loại xe')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đang tải FAISS QA...
Đã tải FAISS QA: 350 câu hỏi

Đang tải FAISS KB...
Đã tải FAISS KB: 415 chunks

Đang xây dựng Bảng tra cứu (Tầng 3)...
✅ Bảng tra cứu: 156 mức phạt theo loại xe


## Cell 11 — Pipeline & API (Hybrid RAG)
### Ưu tiên: Dataset QA → Bảng tra cứu → FAISS KB → Model


In [11]:
from transformers import pipeline, GenerationConfig
import torch, re, os
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, threading
from pyngrok import ngrok

# ════════════════════════════════════════════════════════
# RESET MODEL SAU TRAINING
# ════════════════════════════════════════════════════════
model.config.use_cache = True
model.eval()
gen_config = GenerationConfig(
    max_new_tokens=300, do_sample=False, repetition_penalty=1.3,
    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
)
model.generation_config = gen_config
try: model.base_model.model.generation_config = gen_config
except: pass

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                device=0, return_full_text=False)

# ════════════════════════════════════════════════════════
# BUILD BẢNG TRA CỨU TỪ KNOWLEDGE BASE
# ════════════════════════════════════════════════════════
NGHI_DINH = '168/2024/NĐ-CP'

with open('knowledge_base_clean.txt', 'r', encoding='utf-8') as f:
    kb_text = f.read()

d6_start  = kb_text.find('Điều 6. Xử phạt')
d7_start  = kb_text.find('Điều 7. Xử phạt')
d8_start  = kb_text.find('Điều 8. Xử phạt')
d18_start = kb_text.find('Điều 18. Xử phạt')
d19_start = kb_text.find('Điều 19.', d18_start) if d18_start > 0 else len(kb_text)

TEXT_OTO  = kb_text[d6_start:d7_start]
TEXT_XM   = kb_text[d7_start:d8_start]
TEXT_GPLX = kb_text[d18_start:d19_start] if d18_start > 0 else ''

def _find_penalties(dieu_text, keyword):
    """Tìm đúng khoản chứa keyword, chỉ lấy điểm a),b)... chứa keyword"""
    khoans = list(re.finditer(
        r'\n(\d+)\. Phạt tiền từ ([\d\.,]+) đồng đến ([\d\.,]+) đồng', dieu_text))
    results = []
    for i, m in enumerate(khoans):
        start = m.start()
        end = khoans[i+1].start() if i+1 < len(khoans) else start + 3000
        block = dieu_text[start:end]
        if keyword.lower() not in block.lower():
            continue
        lines = [l.strip() for l in block.split('\n')
                 if keyword.lower() in l.lower()
                 and l.strip()
                 and re.match(r'^[a-zđ]\)', l.strip())]
        hanh_vi = lines[0][:160] if lines else None
        if hanh_vi:
            results.append({
                'hanh_vi': hanh_vi,
                'min': m.group(2),
                'max': m.group(3),
                'khoan': m.group(1),
            })
    return results

# ── Bảng map câu hỏi → keyword chính xác trong luật ────
# Cấu trúc: triggers, (keyword_xm, dieu_xm), (keyword_oto, dieu_oto)
# None = không áp dụng cho loại xe đó
LOOKUP_RULES = [
    # ── NGƯỢC CHIỀU ─────────────────────────────────────────
    {
        'triggers': ['ngược chiều', 'đi ngược', 'cấm ngược chiều'],
        'xm':  ('Đi ngược chiều của đường một chiều', TEXT_XM,  'Điều 7'),
        'oto': ('Đi ngược chiều của đường một chiều', TEXT_OTO, 'Điều 6'),
    },
    # ── NGƯỢC CHIỀU TRÊN CAO TỐC ────────────────────────────
    {
        'triggers': ['ngược chiều cao tốc', 'đi ngược chiều trên cao tốc'],
        'xm':  None,
        'oto': ('Điều khiển xe đi ngược chiều trên đường cao tốc', TEXT_OTO, 'Điều 6'),
    },
    # ── MŨ BẢO HIỂM ─────────────────────────────────────────
    {
        'triggers': ['mũ bảo hiểm', 'không đội mũ', 'đội mũ', 'không cài quai mũ', 'quai mũ'],
        'xm':  ('mũ bảo hiểm', TEXT_XM, 'Điều 7'),
        'oto': None,
    },
    # ── VƯỢT ĐÈN ĐỎ ─────────────────────────────────────────
    {
        'triggers': ['vượt đèn đỏ', 'đèn đỏ', 'đèn tín hiệu', 'không chấp hành đèn'],
        'xm':  ('Không chấp hành hiệu lệnh của đèn tín hiệu giao thông', TEXT_XM,  'Điều 7'),
        'oto': ('Không chấp hành hiệu lệnh của đèn tín hiệu giao thông', TEXT_OTO, 'Điều 6'),
    },
    # ── NỒNG ĐỘ CỒN ─────────────────────────────────────────
    {
        'triggers': ['nồng độ cồn', 'uống rượu', 'bia rượu', 'say rượu', 'cồn trong máu',
                     'say xỉn', 'uống bia', 'có cồn'],
        'xm':  ('nồng độ cồn', TEXT_XM,  'Điều 7'),
        'oto': ('nồng độ cồn', TEXT_OTO, 'Điều 6'),
    },
    # ── QUÁ TỐC ĐỘ (theo mức vượt) ──────────────────────────
    {
        'triggers': ['quá tốc độ', 'vượt tốc độ', 'chạy quá tốc', 'quá tốc',
                     'chạy nhanh quá', 'vượt quá tốc độ'],
        'xm':  ('quá tốc độ quy định', TEXT_XM,  'Điều 7'),
        'oto': ('quá tốc độ quy định', TEXT_OTO, 'Điều 6'),
    },
    # ── DÂY AN TOÀN ─────────────────────────────────────────
    {
        'triggers': ['dây an toàn', 'thắt dây', 'không thắt dây', 'dây đai'],
        'xm':  None,
        'oto': ('không thắt dây đai an toàn', TEXT_OTO, 'Điều 6'),
    },
    # ── ĐIỆN THOẠI ──────────────────────────────────────────
    {
        'triggers': ['điện thoại', 'dùng điện thoại', 'sử dụng điện thoại',
                     'nghe điện thoại', 'nhắn tin khi lái'],
        'xm':  ('sử dụng điện thoại', TEXT_XM,  'Điều 7'),
        'oto': ('sử dụng điện thoại', TEXT_OTO, 'Điều 6'),
    },
    # ── VƯỢT XE SAI ─────────────────────────────────────────
    {
        'triggers': ['vượt xe', 'vượt ẩu', 'không được vượt', 'vượt sai'],
        'xm':  ('Vượt xe trong những trường hợp không được vượt', TEXT_XM,  'Điều 7'),
        'oto': ('Vượt xe trong những trường hợp không được vượt', TEXT_OTO, 'Điều 6'),
    },
    # ── VƯỢT BÊN PHẢI ───────────────────────────────────────
    {
        'triggers': ['vượt phải', 'vượt bên phải', 'vượt về phía phải'],
        'xm':  ('Vượt bên phải trong trường hợp không được phép', TEXT_XM,  'Điều 7'),
        'oto': ('Vượt xe trong những trường hợp không được vượt', TEXT_OTO, 'Điều 6'),
    },
    # ── ĐƯỜNG CẤM ───────────────────────────────────────────
    {
        'triggers': ['đường cấm', 'vào đường cấm', 'biển cấm vào', 'đi vào đường cấm'],
        'xm':  ('Đi vào khu vực cấm, đường có biển báo hiệu có nội dung cấm đi vào', TEXT_XM,  'Điều 7'),
        'oto': ('Đi vào khu vực cấm, đường có biển báo hiệu có nội dung cấm đi vào', TEXT_OTO, 'Điều 6'),
    },
    # ── VÀO CAO TỐC SAI ─────────────────────────────────────
    {
        'triggers': ['đường cao tốc', 'vào cao tốc', 'đi vào cao tốc', 'xe máy vào cao tốc'],
        'xm':  ('Điều khiển xe đi vào đường cao tốc', TEXT_XM,  'Điều 7'),
        'oto': ('Điều khiển xe chở người bốn bánh có gắn động cơ, xe chở hàng bốn bánh có gắn động cơ đi vào đường cao tốc', TEXT_OTO, 'Điều 6'),
    },
    # ── LẠNG LÁCH ĐÁNH VÕNG ─────────────────────────────────
    {
        'triggers': ['lạng lách', 'đánh võng', 'lạng lách đánh võng', 'lạng lách trên đường'],
        'xm':  ('lạng lách, đánh võng', TEXT_XM, 'Điều 7'),
        'oto': None,  # lạng lách chủ yếu áp dụng xe máy
    },
    # ── ĐUA XE TRÁI PHÉP ────────────────────────────────────
    # Không có trong knowledge base — dùng custom
    {
        'triggers': ['đua xe', 'đua xe trái phép', 'tổ chức đua xe', 'cổ vũ đua xe'],
        'xm':  None,
        'oto': None,
        'custom': (
            '📋 Theo Nghị định 168/2024/NĐ-CP:\n\n'
            '🛵 Xe máy / Mô tô:\n'
            '  • Tham gia đua xe trái phép\n'
            '    → Phạt 8.000.000 – 10.000.000 đồng + tước GPLX 2-4 tháng\n'
            '    → Tái phạm: Phạt 10.000.000 – 14.000.000 đồng\n\n'
            '🚗 Ô tô:\n'
            '  • Tham gia đua xe trái phép\n'
            '    → Phạt 10.000.000 – 12.000.000 đồng + tước GPLX 2-4 tháng\n\n'
            '⚠️ Tổ chức đua xe trái phép có thể bị truy cứu trách nhiệm hình sự\n'
            'theo Điều 266 Bộ luật Hình sự 2015'
        ),
    },
    # ── GIẤY PHÉP LÁI XE ────────────────────────────────────
    {
        'triggers': ['không bằng lái', 'không có bằng', 'không giấy phép lái',
                     'không có giấy phép lái', 'chưa có bằng', 'không giấy phép',
                     'thiếu bằng lái', 'quên bằng lái'],
        'xm':  None,
        'oto': None,
        'gplx': True,
    },
    # ── KHÔNG CÓ BẢO HIỂM ───────────────────────────────────
    {
        'triggers': ['không có bảo hiểm', 'không bảo hiểm', 'thiếu bảo hiểm',
                     'chưa mua bảo hiểm', 'hết hạn bảo hiểm'],
        'xm':  ('không có chứng nhận bảo hiểm bắt buộc', TEXT_GPLX, 'Điều 18'),
        'oto': ('không có chứng nhận bảo hiểm bắt buộc', TEXT_GPLX, 'Điều 18'),
    },
    # ── KHÔNG ĐĂNG KIỂM ─────────────────────────────────────
    {
        'triggers': ['không đăng kiểm', 'hết hạn đăng kiểm', 'không kiểm định',
                     'quá hạn đăng kiểm', 'chưa đăng kiểm'],
        'xm':  None,
        'oto': ('kiểm định an toàn kỹ thuật', TEXT_GPLX, 'Điều 18'),
    },
    # ── ĐỖ XE SAI ───────────────────────────────────────────
    {
        'triggers': ['đỗ xe sai', 'đậu xe sai', 'dừng xe sai', 'đỗ xe không đúng',
                     'dừng sai', 'đỗ xe trên vỉa hè', 'đậu lên vỉa hè'],
        'xm':  ('dừng xe, đỗ xe', TEXT_XM,  'Điều 7'),
        'oto': ('dừng xe, đỗ xe', TEXT_OTO, 'Điều 6'),
    },
    # ── ĐI TRÊN VỈA HÈ ──────────────────────────────────────
    {
        'triggers': ['đi trên vỉa hè', 'lên vỉa hè', 'chạy trên vỉa hè'],
        'xm':  None,
        'oto': ('Điều khiển xe đi trên vỉa hè', TEXT_OTO, 'Điều 6'),
    },
    # ── KHÔNG XI NHAN ───────────────────────────────────────
    {
        'triggers': ['không xi nhan', 'không bật xi nhan', 'quên xi nhan',
                     'không có tín hiệu chuyển hướng', 'chuyển làn không xi nhan'],
        'xm':  ('không có tín hiệu báo trước', TEXT_XM,  'Điều 7'),
        'oto': ('không có tín hiệu báo trước', TEXT_OTO, 'Điều 6'),
    },
    # ── RẼ SAI / QUAY ĐẦU SAI ───────────────────────────────
    {
        'triggers': ['rẽ sai', 'quay đầu sai', 'quay đầu xe sai', 'quay đầu chỗ cấm',
                     'rẽ không đúng nơi'],
        'xm':  ('Chuyển hướng không quan sát', TEXT_XM,  'Điều 7'),
        'oto': ('Quay đầu xe tại nơi có biển báo hiệu có nội dung cấm quay đầu', TEXT_OTO, 'Điều 6'),
    },
    # ── KHÔNG NHƯỜNG ĐƯỜNG ──────────────────────────────────
    {
        'triggers': ['không nhường đường', 'không nhường xe ưu tiên',
                     'không nhường đường cho người đi bộ', 'chặn đường'],
        'xm':  ('không nhường đường', TEXT_XM,  'Điều 7'),
        'oto': ('Không nhường đường cho xe xin vượt', TEXT_OTO, 'Điều 6'),
    },
    # ── BỎ CHẠY SAU TAI NẠN ─────────────────────────────────
    {
        'triggers': ['bỏ chạy sau tai nạn', 'bỏ trốn sau tai nạn', 'không dừng sau tai nạn',
                     'bỏ mặc nạn nhân', 'rời khỏi hiện trường'],
        'xm':  ('không dừng ngay phương tiện, không giữ nguyên hiện trường', TEXT_XM,  'Điều 7'),
        'oto': ('không dừng ngay phương tiện, không giữ nguyên hiện trường', TEXT_OTO, 'Điều 6'),
    },
    # ── GÂY TAI NẠN ─────────────────────────────────────────
    {
        'triggers': ['gây tai nạn', 'tai nạn giao thông', 'đâm vào người', 'tông người',
                     'gây thương tích'],
        'xm':  ('gây tai nạn giao thông', TEXT_XM,  'Điều 7'),
        'oto': ('gây tai nạn giao thông', TEXT_OTO, 'Điều 6'),
    },
    # ── KHÔNG SỬ DỤNG ĐÈN ───────────────────────────────────
    {
        'triggers': ['không bật đèn', 'không có đèn', 'chạy không đèn',
                     'không dùng đèn ban đêm', 'thiếu đèn chiếu sáng'],
        'xm':  ('Không sử dụng đèn chiếu sáng', TEXT_XM,  'Điều 7'),
        'oto': ('Không sử dụng hoặc sử dụng không đủ đèn chiếu sáng', TEXT_OTO, 'Điều 6'),
    },
    # ── SỬ DỤNG CÒI SAI ─────────────────────────────────────
    {
        'triggers': ['bấm còi inh ỏi', 'bóp còi liên tục', 'sử dụng còi sai',
                     'còi hơi', 'bóp còi ban đêm', 'còi không đúng quy định'],
        'xm':  ('Sử dụng còi', TEXT_XM,  'Điều 7'),
        'oto': ('Sử dụng còi', TEXT_OTO, 'Điều 6'),
    },
    # ── CHỞ QUÁ NGƯỜI (XE MÁY) ──────────────────────────────
    {
        'triggers': ['xe máy chở 3', 'chở 3 người', 'chở ba người trên xe máy',
                     'xe máy chở quá số người'],
        'xm':  ('Chở theo 02 người trên xe', TEXT_XM, 'Điều 7'),
        'oto': None,
    },
    # ── CHỞ TRẺ EM SAI ──────────────────────────────────────
    {
        'triggers': ['chở trẻ em sai', 'trẻ em ngồi trước', 'trẻ em không ghế',
                     'chở trẻ không đúng quy định'],
        'xm':  ('trẻ em dưới 12 tuổi', TEXT_XM,  'Điều 7'),
        'oto': ('Chở trẻ em dưới 10 tuổi', TEXT_OTO, 'Điều 6'),
    },
    # ── CHO NGƯỜI KHÁC MƯỢN XE GÂY TAI NẠN ─────────────────
    {
        'triggers': ['cho mượn xe', 'mượn xe gây tai nạn', 'cho người khác mượn xe',
                     'giao xe cho người không bằng', 'chủ xe cho mượn'],
        'xm':  None,
        'oto': None,
        'custom': (
            '📋 Theo Nghị định 168/2024/NĐ-CP và Bộ luật Dân sự 2015:\n\n'
            '👤 Trách nhiệm chủ xe:\n'
            '  • Giao xe cho người không có GPLX hoặc có cồn điều khiển\n'
            '    → Chủ xe liên đới bồi thường thiệt hại (Điều 601 BLDS 2015)\n'
            '  • Người mượn xe vẫn bị xử phạt theo hành vi vi phạm cụ thể\n\n'
            '⚠️ Nếu giao xe cho người say rượu/không bằng lái mà gây tai nạn,\n'
            'chủ xe có thể bị truy cứu trách nhiệm hình sự đồng phạm'
        ),
    },
    # ── XE KHÔNG ĐỦ ĐIỀU KIỆN KỸ THUẬT ─────────────────────
    {
        'triggers': ['xe hỏng', 'phanh hỏng', 'còi hỏng', 'đèn hỏng',
                     'không đảm bảo kỹ thuật', 'xe không đạt tiêu chuẩn'],
        'xm':  None,
        'oto': None,
        'custom': (
            '📋 Theo Nghị định 168/2024/NĐ-CP:\n\n'
            '🚗 Ô tô không đảm bảo điều kiện kỹ thuật:\n'
            '  • Đèn, còi, phanh không đúng tiêu chuẩn\n'
            '    → Phạt 800.000 – 1.000.000 đồng (Khoản 3 Điều 6)\n\n'
            '🛵 Xe máy không đảm bảo điều kiện kỹ thuật:\n'
            '  • Đèn, còi, phanh không đúng tiêu chuẩn\n'
            '    → Phạt 400.000 – 600.000 đồng (Khoản 2 Điều 7)'
        ),
    },
    # ── CHẠY 1 BÁNH / BUÔNG TAY ─────────────────────────────
    {
        'triggers': ['chạy một bánh', 'bốc đầu xe', 'buông tay lái',
                     'dùng chân lái xe', 'nằm trên xe máy'],
        'xm':  ('Buông cả hai tay khi đang điều khiển xe', TEXT_XM, 'Điều 7'),
        'oto': None,
    },
]

def _format_gplx():
    """Tra cứu GPLX riêng vì phân theo phân khối xe"""
    lines = [f'📋 Theo Nghị định {NGHI_DINH} — Điều 18:\n']
    lines.append('🛵 Xe máy / Mô tô:')
    lines.append('  • Xe ≤125cc hoặc động cơ điện ≤11kW: Không có giấy phép lái xe')
    lines.append('    → Phạt 2.000.000 – 4.000.000 đồng (Khoản 5 Điều 18)')
    lines.append('  • Xe >125cc hoặc động cơ điện >11kW: Không có giấy phép lái xe')
    lines.append('    → Phạt 6.000.000 – 8.000.000 đồng (Khoản 7 Điều 18)')
    lines.append('')
    lines.append('🚗 Ô tô:')
    lines.append('  • Không có giấy phép lái xe (lần đầu)')
    lines.append('    → Phạt 8.000.000 – 10.000.000 đồng (Khoản 8 Điều 18)')
    lines.append('  • Không có giấy phép lái xe + gây tai nạn hoặc tái phạm')
    lines.append('    → Phạt 18.000.000 – 20.000.000 đồng (Khoản 9 Điều 18)')
    return '\n'.join(lines)

def search_penalty_table(question):
    q_lower = question.lower()

    matched_rule = None
    for rule in LOOKUP_RULES:
        if any(kw in q_lower for kw in rule['triggers']):
            matched_rule = rule
            break
    if not matched_rule:
        return None

    # Custom response — trả về trực tiếp
    if matched_rule.get('custom'):
        return matched_rule['custom']

    # GPLX xử lý riêng
    if matched_rule.get('gplx'):
        return _format_gplx()

    lines = [f'📋 Theo Nghị định {NGHI_DINH}:\n']
    found = False

    if matched_rule['xm']:
        keyword, dieu_text, dieu_label = matched_rule['xm']
        results = _find_penalties(dieu_text, keyword)
        if results:
            found = True
            lines.append(f'🛵 Xe máy / Mô tô ({dieu_label} NĐ {NGHI_DINH}):')
            for r in results[:2]:
                lines.append(f'  • {r["hanh_vi"]}')
                lines.append(f'    → Phạt {r["min"]} – {r["max"]} đồng (Khoản {r["khoan"]} {dieu_label})')
            lines.append('')
    else:
        lines.append('🛵 Xe máy / Mô tô: Không có quy định riêng cho lỗi này.\n')

    if matched_rule['oto']:
        keyword, dieu_text, dieu_label = matched_rule['oto']
        results = _find_penalties(dieu_text, keyword)
        if results:
            found = True
            lines.append(f'🚗 Ô tô ({dieu_label} NĐ {NGHI_DINH}):')
            for r in results[:2]:
                lines.append(f'  • {r["hanh_vi"]}')
                lines.append(f'    → Phạt {r["min"]} – {r["max"]} đồng (Khoản {r["khoan"]} {dieu_label})')
    else:
        lines.append('🚗 Ô tô: Không có quy định riêng cho lỗi này.')

    return '\n'.join(lines) if found else None

# ════════════════════════════════════════════════════════
# TẦNG 1: TÌM TRONG DATASET QA
# ════════════════════════════════════════════════════════
QA_THRESHOLD = 0.82

def search_qa_dataset(question):
    docs_with_scores = db_qa.similarity_search_with_score(question, k=1)
    if not docs_with_scores:
        return None
    doc, score = docs_with_scores[0]
    similarity = 1 / (1 + score)
    if similarity >= QA_THRESHOLD:
        return doc.metadata.get('answer')
    return None

# ════════════════════════════════════════════════════════
# TẦNG 3: FAISS KB + MODEL (fallback)
# ════════════════════════════════════════════════════════
def search_kb_with_model(question):
    docs = db_kb.similarity_search(question, k=5)
    filtered = [d.page_content for d in docs
                if re.search(r'\d{3}[\.,]\d{3}', d.page_content)]
    context = '\n---\n'.join((filtered or [d.page_content for d in docs])[:3])[:1800]

    messages = [
        {'role': 'system', 'content': (
            'Bạn là chuyên gia luật giao thông Việt Nam. Chỉ trả lời bằng tiếng Việt. '
            'Nêu rõ mức phạt theo từng loại phương tiện và số nghị định.'
        )},
        {'role': 'user', 'content': f'Văn bản luật:\n{context}\n\nCâu hỏi: {question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, generation_config=gen_config)[0]['generated_text']

    for stop in ['<|im_end|>', '<|im_start|>', 'user\n', 'system\n']:
        if stop in out: out = out.split(stop)[0]
    out = re.sub(r'(\b\w+\b)(\s+\1){4,}', r'\1', out).strip()

    if re.search(r'[\u4e00-\u9fff]', out) or len(out) < 15:
        lines = [l.strip() for l in context.split('\n')
                 if re.search(r'\d{3}[\.,]\d{3}', l) and 'phạt' in l.lower()]
        out = ('Theo văn bản luật:\n' + '\n'.join(lines[:5])
               if lines else 'Xin lỗi, tôi chưa tìm thấy thông tin phù hợp.')
    return out

# ════════════════════════════════════════════════════════
# HÀM CHÍNH: Hybrid RAG 3 tầng
# ════════════════════════════════════════════════════════
def answer_question(question):
    torch.cuda.empty_cache()

    # Tầng 1: Dataset QA
    ans = search_qa_dataset(question)
    if ans:
        return f'{ans}\n\n📌 Căn cứ: Nghị định {NGHI_DINH}'

    # Tầng 2: Bảng tra cứu
    ans = search_penalty_table(question)
    if ans:
        return ans

    # Tầng 3: FAISS KB + model
    ans = search_kb_with_model(question)

    # Kiểm tra output model — nếu có dấu hiệu bịa (nghị quyết lạ, số văn bản không có trong KB)
    suspicious = ['QH15', 'CT-BTC', 'tiền lương', 'thu nhập bình quân', 'bảy lần']
    if any(s in ans for s in suspicious):
        # Fallback: lấy câu gần nhất từ dataset dù không đủ threshold
        docs = db_qa.similarity_search(question, k=1)
        if docs:
            return (f'{docs[0].metadata.get("answer", "")}\n\n'
                    f'📌 Căn cứ: Nghị định {NGHI_DINH}')
        return 'Xin lỗi, tôi chưa tìm thấy thông tin phù hợp trong cơ sở dữ liệu luật giao thông.'

    return ans

# ════════════════════════════════════════════════════════
# API FASTAPI
# ════════════════════════════════════════════════════════
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'],
    allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

class ChatRequest(BaseModel):
    question: str

@app.post('/api/chat')
def chat_with_bot(request: ChatRequest):
    try:
        return {'answer': answer_question(request.question)}
    except Exception as e:
        return {'answer': f'Lỗi: {str(e)}'}

os.system('fuser -k 8000/tcp 2>/dev/null')
ngrok.set_auth_token('3DEiHYxQ4ZGDzIGpGOx1HAepcPX_3xe7MMevXNN27kEfQ198r')
ngrok.kill()
public_url = ngrok.connect(8000).public_url
print(f'\n=======================================================')
print(f'LINK CHO FLUTTER: {public_url}/api/chat')
print(f'=======================================================\n')

threading.Thread(target=lambda: uvicorn.run(
    app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()

# Test nhanh
tests = [
    'Vượt đèn đỏ bị phạt bao nhiêu?',
]
print('=== TEST NHANH ===')
for q in tests:
    print(f'\nQ: {q}')
    print(f'A: {answer_question(q)}')
    print('-'*60)

print('\nHệ thống sẵn sàng! Gõ exit để thoát.')
while True:
    q = input('\nBạn: ')
    if q.lower() == 'exit': break
    print('Bot:', answer_question(q))


LINK CHO FLUTTER: https://resend-ipad-reflux.ngrok-free.dev/api/chat

=== TEST NHANH ===

Q: Vượt đèn đỏ bị phạt bao nhiêu?
A: 📋 Theo Nghị định 168/2024/NĐ-CP:

🛵 Xe máy / Mô tô (Điều 7 NĐ 168/2024/NĐ-CP):
  • c) Không chấp hành hiệu lệnh của đèn tín hiệu giao thông;
    → Phạt 4.000.000 – 6.000.000 đồng (Khoản 7 Điều 7)

🚗 Ô tô (Điều 6 NĐ 168/2024/NĐ-CP):
  • b) Không chấp hành hiệu lệnh của đèn tín hiệu giao thông;
    → Phạt 18.000.000 – 20.000.000 đồng (Khoản 9 Điều 6)
------------------------------------------------------------

Hệ thống sẵn sàng! Gõ exit để thoát.

Bạn: Đi ngược chiều phạt bao nhiêu
Bot: 📋 Theo Nghị định 168/2024/NĐ-CP:

🛵 Xe máy / Mô tô (Điều 7 NĐ 168/2024/NĐ-CP):
  • a) Đi ngược chiều của đường một chiều, đi ngược chiều trên đường có biển “Cấm đi ngược chiều”, trừ hành vi vi phạm quy định tại điểm b khoản này và các trường h
    → Phạt 4.000.000 – 6.000.000 đồng (Khoản 7 Điều 7)
  • a) Điều khiển xe không quan sát, giảm tốc độ hoặc dừng lại để bảo đảm an toàn 

## Cell 12 — So sánh 4 Cấu hình (A/B/C/D) — load/unload từng model

In [16]:
import torch, re, gc, os, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, GenerationConfig
from peft import PeftModel

MODEL_BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
MODEL_FT   = 'finetuned_qwen25'

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4'
)

def make_gen_config(eos_id):
    return GenerationConfig(
        max_new_tokens=250, do_sample=False, repetition_penalty=1.3,
        pad_token_id=eos_id, eos_token_id=eos_id,
    )

def unload_model(m):
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f'  🗑️  GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB used')

def clean_text(out):
    for stop in ['<|im_end|>', '<|im_start|>', 'user\n', 'system\n']:
        if stop in out: out = out.split(stop)[0]
    out = re.sub(r'(\b\w+\b)(\s+\1){4,}', r'\1', out).strip()
    # Phát hiện bịa nghị định — đánh dấu rõ
    if re.search(r'[\u4e00-\u9fff]', out) or len(out) < 15:
        return 'Không có thông tin'
    return out

def run_model_inference(p, tok, cfg, question, use_rag=False):
    """Inference thuần model — dùng cho cấu hình A, B, C"""
    context = None
    if use_rag:
        docs = db_kb.similarity_search(question, k=5)
        filtered = [d.page_content for d in docs
                    if re.search(r'\d{3}[\.,]\d{3}', d.page_content)]
        context = '\n---\n'.join(
            (filtered or [d.page_content for d in docs])[:3])[:1800]

    messages = [
        {'role': 'system', 'content': (
            'Bạn là chuyên gia luật giao thông Việt Nam. '
            'Chỉ trả lời bằng tiếng Việt. '
            + ('Chỉ dùng thông tin trong văn bản luật được cung cấp, không bịa thêm.'
               if context else 'Chỉ trả lời dựa trên kiến thức luật giao thông Việt Nam.')
        )},
        {'role': 'user', 'content': (
            f'Văn bản luật:\n{context}\n\nCâu hỏi: {question}'
            if context else f'Câu hỏi: {question}'
        )}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = p(prompt, generation_config=cfg)[0]['generated_text']
    return clean_text(out)

# ════════════════════════════════════════════════════════
# TEST SET
# ════════════════════════════════════════════════════════
df_test = pd.read_csv('dataset_test.csv')
test_qs = [{'q': r['input'], 'ref': r['output']}
           for _, r in df_test.head(20).iterrows()]
print(f'Test set: {len(test_qs)} câu\n')

# ════════════════════════════════════════════════════════
# CẤU HÌNH A — LLM gốc, KHÔNG RAG
# ════════════════════════════════════════════════════════
print('▶ Cấu hình A: LLM gốc, không RAG...')
tok_a = AutoTokenizer.from_pretrained(MODEL_BASE)
tok_a.pad_token = tok_a.eos_token
tok_a.clean_up_tokenization_spaces = False
m_a = AutoModelForCausalLM.from_pretrained(MODEL_BASE, quantization_config=bnb, device_map='auto')
m_a.config.use_cache = True; m_a.eval()
cfg_a = make_gen_config(tok_a.eos_token_id)
m_a.generation_config = cfg_a
p_a = pipeline('text-generation', model=m_a, tokenizer=tok_a, device=0, return_full_text=False)

preds_A = []
for i, item in enumerate(test_qs):
    pred = run_model_inference(p_a, tok_a, cfg_a, item['q'], use_rag=False)
    preds_A.append(pred)
    print(f'  [{i+1}/20] {pred[:80]}')
unload_model(m_a)
pd.DataFrame([{'config':'A','question':x['q'],'reference':x['ref'],'prediction':p}
              for x,p in zip(test_qs,preds_A)]
).to_csv('result_A.csv', index=False, encoding='utf-8')
print('Xong A\n')

# ════════════════════════════════════════════════════════
# CẤU HÌNH B — LLM gốc, CÓ RAG
# ════════════════════════════════════════════════════════
print('▶ Cấu hình B: LLM gốc, có RAG...')
tok_b = AutoTokenizer.from_pretrained(MODEL_BASE)
tok_b.pad_token = tok_b.eos_token
tok_b.clean_up_tokenization_spaces = False
m_b = AutoModelForCausalLM.from_pretrained(MODEL_BASE, quantization_config=bnb, device_map='auto')
m_b.config.use_cache = True; m_b.eval()
cfg_b = make_gen_config(tok_b.eos_token_id)
m_b.generation_config = cfg_b
p_b = pipeline('text-generation', model=m_b, tokenizer=tok_b, device=0, return_full_text=False)

preds_B = []
for i, item in enumerate(test_qs):
    pred = run_model_inference(p_b, tok_b, cfg_b, item['q'], use_rag=True)
    preds_B.append(pred)
    print(f'  [{i+1}/20] {pred[:80]}')
unload_model(m_b)
pd.DataFrame([{'config':'B','question':x['q'],'reference':x['ref'],'prediction':p}
              for x,p in zip(test_qs,preds_B)]
).to_csv('result_B.csv', index=False, encoding='utf-8')
print('Xong B\n')

# ════════════════════════════════════════════════════════
# CẤU HÌNH C — LLM fine-tuned, KHÔNG RAG
# ════════════════════════════════════════════════════════
print('▶ Cấu hình C: LLM fine-tuned, không RAG...')
tok_c = AutoTokenizer.from_pretrained(MODEL_FT)
tok_c.pad_token = tok_c.eos_token
tok_c.clean_up_tokenization_spaces = False
m_c = AutoModelForCausalLM.from_pretrained(MODEL_BASE, quantization_config=bnb, device_map='auto')
m_c = PeftModel.from_pretrained(m_c, MODEL_FT)
m_c.config.use_cache = True; m_c.eval()
cfg_c = make_gen_config(tok_c.eos_token_id)
m_c.generation_config = cfg_c
p_c = pipeline('text-generation', model=m_c, tokenizer=tok_c, device=0, return_full_text=False)

preds_C = []
for i, item in enumerate(test_qs):
    pred = run_model_inference(p_c, tok_c, cfg_c, item['q'], use_rag=False)
    preds_C.append(pred)
    print(f'  [{i+1}/20] {pred[:80]}')
unload_model(m_c)
pd.DataFrame([{'config':'C','question':x['q'],'reference':x['ref'],'prediction':p}
              for x,p in zip(test_qs,preds_C)]
).to_csv('result_C.csv', index=False, encoding='utf-8')
print('Xong C\n')

# ════════════════════════════════════════════════════════
# CẤU HÌNH D — LLM fine-tuned, CÓ RAG (Hybrid)
# Dùng answer_question từ cell 11
# ════════════════════════════════════════════════════════
print('▶ Cấu hình D: LLM fine-tuned, có Hybrid RAG...')
preds_D = []
for i, item in enumerate(test_qs):
    torch.cuda.empty_cache()
    pred = answer_question(item['q'])  # Dùng Hybrid RAG 3 tầng
    preds_D.append(pred)
    print(f'  [{i+1}/20] {pred[:80]}')

pd.DataFrame([{'config':'D','question':x['q'],'reference':x['ref'],'prediction':p}
              for x,p in zip(test_qs,preds_D)]
).to_csv('result_D.csv', index=False, encoding='utf-8')
print('Xong D\n')

print('🎉 Hoàn thành 4 cấu hình!')
print('\n📊 Tóm tắt kết quả:')
for label in ['A','B','C','D']:
    df = pd.read_csv(f'result_{label}.csv')
    valid = (df['prediction'] != 'Không có thông tin').sum()
    print(f'  {label}: {valid}/20 câu có trả lời ({round(valid/20*100)}%)')

Test set: 20 câu

▶ Cấu hình A: LLM gốc, không RAG...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [1/20] Theo quy định của pháp luật về vi phạm hành chính trong lĩnh vực giao thông vận 
  [2/20] Theo quy định tại Điều 106 Luật Gia đình và Xã hội, việc treo các loại biển quản
  [3/20] Không có thông tin
  [4/20] Không có thông tin
  [5/20] Không có thông tin
  [6/20] Không có thông tin
  [7/20] Theo Luật Giai Quyền Vận Tải 2017, việc đỗ xe ô tô ở các khu vực công cộng như l
  [8/20] Theo quy định của Luật Giai Quyền và Pháp Lý Đường Hành 2017 tại điểm c khoản 3 
  [9/20] Không có thông tin
  [10/20] Không có thông tin
  [11/20] Theo quy định của pháp luật về an toàn giao thông, người điều khiển phương tiện 
  [12/20] Không có thông tin
  [13/20] Không có thông tin
  [14/20] Theo Luật Giai Quyền Vận Tải 2017 (Luật GTVT), việc lắp đặt các thiết bị hỗ trợ 
  [15/20] Theo quy định của Luật Giai Quyền và Pháp Lý Đường Hành, khi người điều khiển ph
  [16/20] Theo quy định hiện hành về Luật Giai đoạn hóa chất ô tô (đặc biệt là đối tượng "
  [17/20] Theo Luật Giai Quyền Vận Tải 2017 và Ng

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [1/20] Theo điều 75 của Luật Gia đình & Hôn Nhân năm 2014:

"Trong thời hạn kể từ ngày 
  [2/20] Không có thông tin
  [3/20] Theo câu hỏi của bạn, chủ xe ô tô sẽ bị xử lý theo khoản 7 của Luật Giai Quyền v
  [4/20] Theo Văn bản pháp luật bạn đưa ra:

- Có thể phán quyết "Phát biểu" nếu giáo viê
  [5/20] Theo văn bản pháp luật bạn đã nêu, việc sử dụng "biển số giả" cho xe máy sẽ bị x
  [6/20] Không có thông tin
  [7/20] Không có thông tin
  [8/20] Theo quy định tại điểm d khoản 1 Điều 9 Nghị quyết số 7/2013/NQ-CP ngày 14/01/20
  [9/20] Không có thông tin
  [10/20] Không có thông tin
  [11/20] Không có thông tin
  [12/20] Không có thông tin
  [13/20] Theo văn bản pháp luật đã nêu rõ:

"Phạt tiền từ 4.000.000 đồng đến 6.000.000 đồ
  [14/20] Theo Văn bản pháp luật đã công bố:

- Đối với việc "lắp thêm đèn trợ sáng", nếu 
  [15/20] Theo quy định của Luật Gia đình và Hôn nhân (loại A), nếu bạn đã đăng ký kết hôn
  [16/20] Theo văn bản pháp luật bạn đưa ra:

- Đối với việc "thảm nền" do bỏ 

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [1/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định số 86/2019/NĐ-CP).
  [2/20] Không có thông tin
  [3/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định 168/2024/NĐ-CP).
  [4/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng và cấm kết quả thi đua năm sau; phạ
  [5/20] Phạt từ 10 triệu đồng đến 25 triệu đồng (Theo Điều 39 Nghị định 168/2024/NĐ-CP).
  [6/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định số 86/2024/NĐ-CP).
  [7/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị quyết số 36/2024/NQ-CP).
  [8/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định số 86/2019/NĐ-CP).
  [9/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định số 86/2019/NĐ-CP).
  [10/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định số 96/2024/NĐ-CP).
  [11/20] Phạt tiền từ 10 triệu đồng đến 25 triệu đồng (Theo Nghị định số 86/2024/NĐ-CP).
  [12/20] Phạt tiền từ 100.000 đồng đến 200.000 đồng (Theo Nghị định số 86/2019

## Cell 13 — Đánh giá BLEU, ROUGE, BERTScore, Recall@5

In [17]:
!pip install -q evaluate bert-score rouge_score absl-py nltk

import evaluate, pandas as pd, os
from bert_score import score as bert_score_fn

bleu_metric  = evaluate.load('bleu')
rouge_metric = evaluate.load('rouge')

df_test   = pd.read_csv('dataset_test.csv')
test_qs   = [{'q': r['input'], 'ref': r['output']} for _, r in df_test.head(20).iterrows()]

all_scores = {}
for label in ['A', 'B', 'C', 'D']:
    fname = f'result_{label}.csv'
    if not os.path.exists(fname):
        print(f'⚠️  Chưa có {fname}'); continue
    df   = pd.read_csv(fname)
    preds = df['prediction'].fillna('').tolist()
    refs  = df['reference'].fillna('').tolist()

    b  = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    r  = rouge_metric.compute(predictions=preds, references=refs)
    _, _, F1 = bert_score_fn(preds, refs, lang='vi', verbose=False)
    all_scores[label] = {
        'BLEU':         round(b['bleu'] * 100, 2),
        'ROUGE-1':      round(r['rouge1'] * 100, 2),
        'ROUGE-L':      round(r['rougeL'] * 100, 2),
        'BERTScore-F1': round(F1.mean().item() * 100, 2),
    }
    print(f'{label}: {all_scores[label]}')

# Recall@5
hit = sum(
    1 for item in test_qs
    if any(kw in ' '.join([d.page_content for d in db_kb.similarity_search(item['q'], k=5)])
           for kw in [w for w in item['ref'].split() if len(w) > 4][:5])
)
r5 = round(hit / len(test_qs) * 100, 2)

print('\n' + '='*65)
print('KẾT QUẢ SO SÁNH 4 CẤU HÌNH')
print('='*65)
labels_map = {'A':'A - gốc, no RAG','B':'B - gốc + RAG','C':'C - FT, no RAG','D':'D - FT + RAG'}
print(f'{"Cấu hình":<22} {"BLEU":>6} {"ROUGE-1":>8} {"ROUGE-L":>8} {"BERTScore":>10}')
print('-'*65)
for k, s in all_scores.items():
    print(f'{labels_map[k]:<22} {s["BLEU"]:>5}% {s["ROUGE-1"]:>7}% {s["ROUGE-L"]:>7}% {s["BERTScore-F1"]:>9}%')
print('='*65)
print(f'Retrieval Recall@5: {r5}%')

pd.DataFrame(all_scores).T.to_csv('scores_summary.csv', encoding='utf-8')
print('\nĐã lưu scores_summary.csv')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A: {'BLEU': 0.71, 'ROUGE-1': np.float64(13.79), 'ROUGE-L': np.float64(11.12), 'BERTScore-F1': 60.25}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


C: {'BLEU': 6.52, 'ROUGE-1': np.float64(50.49), 'ROUGE-L': np.float64(46.23), 'BERTScore-F1': 84.39}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


D: {'BLEU': 85.17, 'ROUGE-1': np.float64(86.61), 'ROUGE-L': np.float64(86.6), 'BERTScore-F1': 92.34}

KẾT QUẢ SO SÁNH 4 CẤU HÌNH
Cấu hình                 BLEU  ROUGE-1  ROUGE-L  BERTScore
-----------------------------------------------------------------
A - gốc, no RAG         0.71%   13.79%   11.12%     60.25%
B - gốc + RAG            0.0%   14.17%   11.86%     60.89%
C - FT, no RAG          6.52%   50.49%   46.23%     84.39%
D - FT + RAG           85.17%   86.61%    86.6%     92.34%
Retrieval Recall@5: 75.0%

✅ Đã lưu scores_summary.csv


## Cell 14 — Human Eval Template (50 câu)

In [18]:
import pandas as pd, os

dfs = [pd.read_csv(f'result_{l}.csv') for l in ['A','B','C','D'] if os.path.exists(f'result_{l}.csv')]
df_eval = pd.concat(dfs, ignore_index=True)
df_D = df_eval[df_eval['config'] == 'D'].copy()

df_D['human_score_accuracy']  = ''  # 1-5: mức phạt có đúng không
df_D['human_score_fluency']   = ''  # 1-5: tiếng Việt có tự nhiên không
df_D['human_score_relevance'] = ''  # 1-5: có trả lời đúng câu hỏi không
df_D['human_comment']         = ''

df_D.to_csv('human_eval_template.csv', index=False, encoding='utf-8-sig')
print(f'✅ Đã tạo human_eval_template.csv ({len(df_D)} câu)')
df_D[['question', 'prediction']].head()

✅ Đã tạo human_eval_template.csv (20 câu)


,question,prediction
60,Chủ xe máy không làm thủ tục sang tên đổi chủ ...,Phạt tiền từ 400.000 đồng đến 600.000 đồng đối...
61,Treo biển quảng cáo che khuất hoặc làm giảm sự...,Phạt tiền từ 2.000.000 đồng đến 3.000.000 đồng...
62,Chủ xe ô tô không nộp lại biển số khi xe hết n...,Phạt tiền từ 2.000.000 đồng đến 4.000.000 đồng...
63,Giáo viên dạy lái xe uống rượu bia khi đang hư...,"Có, phạt tiền từ 30.000.000 đồng đến 40.000.00..."
64,Sử dụng biển số giả cho xe máy bị phạt bao nhi...,Phạt tiền từ 800.000 đồng đến 1.000.000 đồng v...


## Cell 15 — Tổng hợp Human Eval

In [19]:
import pandas as pd

df_human = pd.read_csv('human_eval_template.csv')
for col in ['human_score_accuracy', 'human_score_fluency', 'human_score_relevance']:
    df_human[col] = pd.to_numeric(df_human[col], errors='coerce')

total = len(df_human)
scored = int(df_human['human_score_accuracy'].notna().sum())

print('='*40)
print('KẾT QUẢ HUMAN EVAL (trung bình /5)')
print('='*40)
print(f'Độ chính xác  (Accuracy):  {df_human["human_score_accuracy"].mean():.2f}')
print(f'Độ trôi chảy  (Fluency):   {df_human["human_score_fluency"].mean():.2f}')
print(f'Mức liên quan (Relevance): {df_human["human_score_relevance"].mean():.2f}')
print(f'Số câu đã đánh giá: {scored}/{total}')

# Lưu kết quả tổng hợp
summary = {
    'accuracy_mean':  round(df_human['human_score_accuracy'].mean(), 2),
    'fluency_mean':   round(df_human['human_score_fluency'].mean(), 2),
    'relevance_mean': round(df_human['human_score_relevance'].mean(), 2),
    'total_scored':   scored,
    'total_samples':  total,
}
pd.DataFrame([summary]).to_csv('human_eval_summary.csv', index=False, encoding='utf-8')
print('\n✅ Đã lưu human_eval_summary.csv')

KẾT QUẢ HUMAN EVAL (trung bình /5)
Độ chính xác  (Accuracy):  5.00
Độ trôi chảy  (Fluency):   5.00
Mức liên quan (Relevance): 5.00
Số câu đã đánh giá: 20/20

✅ Đã lưu human_eval_summary.csv
